# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sumit07-git/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1

The paper reports an observed relationship between search/content signals
and performance outcomes. The important methodology question is how the
label was constructed and whether the validation design prevents future
information from entering the features.

If the label is derived from a future outcome window, the validation design
should preserve that temporal ordering. A random split could place
observations from similar time periods in both training and evaluation,
which may make the result look stronger than it would be at a future
decision point.

### Finding 2

The paper also discusses relationships between content/search signals and
observed performance. A second methodology question is whether the
reported association should be interpreted as prediction or causation.

For my capstone, I will treat model performance as measured predictive
performance on the evaluation data. I will not interpret feature
importance or associations as evidence that changing a feature causes a
search-performance change.

In [2]:
!git clone https://github.com/Sumit07-git/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 149 (delta 60), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 1.90 MiB | 9.24 MiB/s, done.
Resolving deltas: 100% (60/60), done.


In [3]:
!find flyrank-ml-internship -name "content_refresh_anonymized.csv"

flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


In [6]:
df["is_declining"] = (
    df["trend_direction"] == "down"
).astype(int)

print(df["is_declining"].value_counts())

is_declining
1    16262
0    13738
Name: count, dtype: int64


In [7]:
features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "engagement_rate"
]

target = "is_declining"

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nTarget distribution:")
display(df["is_declining"].value_counts())


Rows: 30000
Columns: 45

Target distribution:


,count
is_declining,
1,16262
0,13738


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split (before/after)

In ML-08, I used a random train/test split. For this validation audit, I
use a grouped split by client so that observations from the same client
are not present in both training and test data.

This provides a stricter check of whether the model generalizes to
clients that were not used during training.

I compare the model's Precision@K before and after the stricter split.

### Interpretation

The Random Forest model was evaluated using both the ML-08 random split
and the stricter grouped-by-client split.

Under the random split, Precision@10 was 0.80, Precision@20 was 0.70,
and Precision@50 was 0.82.

Under the grouped-by-client split, Precision@10 increased to 1.00 and
Precision@20 increased to 0.85, while Precision@50 decreased to 0.72.

The results therefore vary by ranking depth. The grouped split performed
better at K=10 and K=20 but worse at K=50. This indicates that the measured
performance is not uniformly better or worse under the stricter split.

These results are directional and specific to this dataset and validation
design. They should be treated as decision-support evidence rather than
evidence that the model causes changes in search performance.

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Random split — same approach used in ML-08
X = df[features]
y = df[target]

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

random_model.fit(X_train_random, y_train_random)

random_scores = random_model.predict_proba(
    X_test_random
)[:, 1]

random_test = X_test_random.copy()
random_test["actual"] = y_test_random.to_numpy()
random_test["model_score"] = random_scores

In [14]:
random_results = []

for k in [10, 20, 50]:
    random_results.append({
        "K": k,
        "ML-08 Random Split Precision@K": precision_at_k(
            random_test,
            "model_score",
            "actual",
            k
        )
    })

random_results_df = pd.DataFrame(random_results)

display(random_results_df)

,K,ML-08 Random Split Precision@K
0,10,0.80
1,20,0.70
2,50,0.82


In [15]:
grouped_results_df = pd.DataFrame([
    {
        "K": k,
        "ML-09 Grouped Split Precision@K": precision_at_k(
            grouped_test,
            "model_score",
            target,
            k
        )
    }
    for k in [10, 20, 50]
])

before_after = random_results_df.merge(
    grouped_results_df,
    on="K"
)

display(before_after)

,K,ML-08 Random Split Precision@K,ML-09 Grouped Split Precision@K
0,10,0.80,1.00
1,20,0.70,0.85
2,50,0.82,0.72


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Final feature set
features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "engagement_rate"
]

target = "is_declining"

# Check whether target or outcome-derived columns are features
leakage_columns = set(features).intersection({
    target,
    "trend_direction",
    "trend_pct"
})

print("Potential direct leakage columns:", leakage_columns)


Potential direct leakage columns: set()


In [17]:
print("FINAL FEATURES:")
for feature in features:
    print(" -", feature)

print("\nTARGET:")
print(" -", target)

print("\nEXCLUDED OUTCOME FIELDS:")
print(" - trend_direction")
print(" - trend_pct")

FINAL FEATURES:
 - impressions_90d
 - clicks_90d
 - sessions_90d
 - avg_position
 - engagement_rate

TARGET:
 - is_declining

EXCLUDED OUTCOME FIELDS:
 - trend_direction
 - trend_pct


In [18]:
correlation_check = df[features + [target]].corr(numeric_only=True)

display(
    correlation_check[target]
    .sort_values(ascending=False)
)

,is_declining
is_declining,1.000000
engagement_rate,-0.012743
impressions_90d,-0.018175
sessions_90d,-0.023141
avg_position,-0.029035
clicks_90d,-0.039680


In [19]:
excluded_fields = [
    "trend_direction",
    "trend_pct"
]

used_excluded_fields = [
    col for col in excluded_fields
    if col in features
]

print("Excluded fields accidentally used as features:")
print(used_excluded_fields)

Excluded fields accidentally used as features:
[]


In [20]:
leakage_audit = pd.DataFrame({
    "Check": [
        "Target included in features",
        "trend_direction included",
        "trend_pct included",
        "Final feature count"
    ],
    "Result": [
        target in features,
        "trend_direction" in features,
        "trend_pct" in features,
        len(features)
    ]
})

display(leakage_audit)

,Check,Result
0,Target included in features,False
1,trend_direction included,False
2,trend_pct included,False
3,Final feature count,5


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

### Original claim

The Random Forest model identifies which pages need to be refreshed.

### Safer claim

On the evaluated dataset, the Random Forest model produced a ranked
estimate of content items associated with the observed `down` trend label.

The model showed measured Precision@K performance under both random and
grouped-by-client validation. The grouped validation produced higher
Precision@10 and Precision@20, but lower Precision@50 than the random
split.

These findings are directional and specific to this dataset and
validation design. The model can be used as decision-support for
prioritizing content review, but the results do not establish that the
model's features cause search-performance changes or that the model
represents Google's ranking system.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

claim_audit = pd.DataFrame({
    "Claim element": [
        "What was observed",
        "What was measured",
        "How it should be used",
        "What it does not prove"
    ],
    "Safe wording": [
        "The model ranked content associated with the observed down-trend label.",
        "Precision@K was measured under random and grouped-by-client splits.",
        "Directional decision-support for prioritizing content review.",
        "It does not prove causation or reproduce Google's ranking system."
    ]
})

display(claim_audit)


,Claim element,Safe wording
0,What was observed,The model ranked content associated with the o...
1,What was measured,Precision@K was measured under random and grou...
2,How it should be used,Directional decision-support for prioritizing ...
3,What it does not prove,It does not prove causation or reproduce Googl...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.